# 2402-00 · Create Environment
### Lab 2402 · One bad query, three symptoms

Builds the data for the lab. Your lab notebook runs it with `%run`, so you only open it to read it.

| What | Where |
|---|---|
| Catalog and schema | `lab2402.shop` |
| Four tables | `customers`, `products`, `orders`, `order_items`: an online shop, 2023–2024 |
| Helpers | `file_stats`, `run_fresh`, `benchmark`, `compare` |

The tables are built on purpose so that **one ordinary query** shows the three classic symptoms in the Query Profile:

1. **No pruning:** every file of `orders` and `order_items` holds dates from both years.
2. **A heavy shuffle:** the dimensions are far too big to broadcast, so the three joins are **shuffle joins** and tens of millions of rows of `order_items` travel.
3. **Skew:** **guest checkout** puts 25% of the orders under one placeholder customer, `customer_idx = 0`, and 1% of the customers (the *whales*) place 80% of the rest.

`order_items` carries the date of its order (`order_date`), as fact tables usually do: the last step of the lab uses it to filter `order_items` **before** the shuffle.

AQE and Photon stay on. Building takes a few minutes the first time; later runs reuse the tables (`rebuild = no`). Set `rebuild = yes` to start again after the lab has clustered `orders` and `order_items`.

## 0. Configuration

| Scale | customers | products | orders | order_items (≈) | files (c / p / o / oi) | Use |
|---|---|---|---|---|---|---|
| `small` | 500 K | 500 K | 1 M | 5.5 M | 4 / 4 / 8 / 8 | smoke test only: the dimensions may be broadcast |
| `medium` | 5 M | 5 M | 10 M | 55 M | 16 / 16 / 32 / 32 | **default** |
| `large` | 15 M | 15 M | 30 M | 165 M | 48 / 48 / 96 / 96 | more pressure |

`guest_share` is the share of orders placed **without an account**. All of them carry the placeholder customer `customer_idx = 0`.

In [ ]:
from pyspark.sql import functions as F
import time, uuid

DEFAULTS = {"scale": "medium", "guest_share": "0.25", "rebuild": "no"}
try:
    dbutils.widgets.dropdown("scale", DEFAULTS["scale"], ["small", "medium", "large"], "Scale")
    dbutils.widgets.text("guest_share", DEFAULTS["guest_share"], "Guest share")
    dbutils.widgets.dropdown("rebuild", DEFAULTS["rebuild"], ["no", "yes"], "Rebuild")
    params = {k: dbutils.widgets.get(k) for k in DEFAULTS}
except NameError:
    params = dict(DEFAULTS)

# Literal names, the same as in the lab's queries
CATALOG, SCHEMA = "lab2402", "shop"
TABLES = ["customers", "products", "orders", "order_items"]
def fq(t): return f"{CATALOG}.{SCHEMA}.{t}"

SCALES = {
    "small":  dict(customers=500_000,    products=500_000,    orders=1_000_000,  files=dict(customers=4,  products=4,  orders=8,  order_items=8),  layout_file_size="1mb"),
    "medium": dict(customers=5_000_000,  products=5_000_000,  orders=10_000_000, files=dict(customers=16, products=16, orders=32, order_items=32), layout_file_size="4mb"),
    "large":  dict(customers=15_000_000, products=15_000_000, orders=30_000_000, files=dict(customers=48, products=48, orders=96, order_items=96), layout_file_size="8mb"),
}
CFG = SCALES[params["scale"]]
LAYOUT_FILE_SIZE = CFG["layout_file_size"]   # used by the layout fix of the lab: small files = fine-grained pruning (demo only)
GUEST_SHARE = float(params["guest_share"])
WHALE_SHARE = 0.80                           # of the registered orders
REBUILD = params["rebuild"] == "yes"
SEED = 42
print(f"scale={params['scale']} · {CFG['orders']:,} orders · guest share {GUEST_SHARE:.0%} · rebuild={REBUILD}")

## 1. Catalog and schema

In [ ]:
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
except Exception as e:   # no CREATE CATALOG privilege: fine if it already exists
    print(f"CREATE CATALOG skipped ({type(e).__name__}); using the existing catalog {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Ready:", f"{CATALOG}.{SCHEMA}")

## 2. The four tables

| Table | Columns | Built in on purpose |
|---|---|---|
| `customers` | `customer_idx`, `customer_id`, `name`, `tier`, `country`, … | `customer_idx = 0` is the **guest** placeholder (tier `guest`); ids 1 … 1% are the **whales** (tier `platinum`) |
| `products` | `product_idx`, `product_id`, `category`, `list_price`, … | wide enough that it is not broadcast at `medium` |
| `orders` | `order_idx`, `order_id`, `customer_idx`, `order_date`, `status`, `channel` | dates random per order: **every file spans 2023–2024** |
| `order_items` | `order_idx`, `line_no`, `product_idx`, `quantity`, `order_date` | 1–10 lines per order; `order_date` copied from its order: **every file spans 2023–2024** too |

The values come from a hash with a fixed seed: every rebuild produces exactly the same rows.

In [ ]:
def u(col, salt):
    """Deterministic uniform value in [0, 1) from a column, the seed and a salt."""
    h = F.xxhash64(F.col(col), F.lit(SEED), F.lit(salt))
    return F.pmod(h, F.lit(1_000_000_007)) / F.lit(1_000_000_007.0)

def pick(values, col, salt):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.floor(u(col, salt) * len(values)) + 1).cast("int"))

N_CUST, N_PROD, N_ORD, FILES = CFG["customers"], CFG["products"], CFG["orders"], CFG["files"]
N_WHALES = max(1, N_CUST // 100)

def ids(n, files, name):
    # one range partition per output file
    return spark.range(0, n, 1, files).withColumnRenamed("id", name)

def build_tables():
    customers = (ids(N_CUST, FILES["customers"], "customer_idx")
        .withColumn("customer_id", F.format_string("C%09d", "customer_idx"))
        .withColumn("name", F.concat(pick(["Ana", "Bram", "Chen", "Daan", "Eva", "Femke", "Joost", "Lotte"], "customer_idx", 1),
                                     F.lit(" "), pick(["de Vries", "Jansen", "Bakker", "Visser", "Smit", "Meijer"], "customer_idx", 2)))
        .withColumn("tier", F.when(F.col("customer_idx") == 0, F.lit("guest"))
                             .when(F.col("customer_idx") <= N_WHALES, F.lit("platinum"))
                             .otherwise(pick(["bronze", "bronze", "silver", "gold"], "customer_idx", 3)))
        .withColumn("country", pick(["NL", "BE", "DE", "FR", "ES", "UK"], "customer_idx", 4))
        .withColumn("email", F.concat(F.lit("user"), F.col("customer_idx").cast("string"), F.lit("@example.com")))
        .withColumn("loyalty_token", F.substring(F.sha2(F.col("customer_id"), 256), 1, 20)))

    categories = ["Electronics", "Home", "Garden", "Toys", "Sports", "Books", "Beauty", "Grocery", "Fashion", "Auto", "Office", "Pets"]
    products = (ids(N_PROD, FILES["products"], "product_idx")
        .withColumn("product_id", F.format_string("P%09d", "product_idx"))
        .withColumn("category", pick(categories, "product_idx", 11))
        .withColumn("name", F.concat_ws(" ", F.col("category"), F.lit("item"), F.col("product_id")))
        .withColumn("list_price", F.round(F.lit(1.0) + u("product_idx", 12) * 499, 2).cast("decimal(10,2)"))
        .withColumn("description", F.substring(F.sha2(F.col("product_id"), 256), 1, 12)))

    r = u("order_idx", 20)
    orders = (ids(N_ORD, FILES["orders"], "order_idx")
        .withColumn("order_id", F.format_string("O%010d", "order_idx"))
        .withColumn("customer_idx",
            F.when(r < GUEST_SHARE, F.lit(0))                                                    # guest checkout
             .when(r < GUEST_SHARE + (1 - GUEST_SHARE) * WHALE_SHARE, 1 + F.floor(u("order_idx", 21) * N_WHALES))
             .otherwise(1 + F.floor(u("order_idx", 22) * (N_CUST - 1))).cast("bigint"))
        .withColumn("order_date", F.date_add(F.lit("2023-01-01").cast("date"), F.floor(u("order_idx", 23) * 731).cast("int")))
        .withColumn("status", pick(["delivered", "delivered", "delivered", "shipped", "returned", "cancelled"], "order_idx", 24))
        .withColumn("channel", pick(["web", "app", "store"], "order_idx", 25)))

    order_items = (ids(N_ORD, FILES["order_items"], "order_idx")
        .withColumn("order_date", F.date_add(F.lit("2023-01-01").cast("date"), F.floor(u("order_idx", 23) * 731).cast("int")))   # same date as its order
        .withColumn("line_no", F.explode(F.sequence(F.lit(1), (F.floor(u("order_idx", 31) * 10) + 1).cast("int"))))
        .withColumn("line_key", F.col("order_idx") * 100 + F.col("line_no"))
        .withColumn("product_idx", F.floor(u("line_key", 32) * N_PROD).cast("bigint"))
        .withColumn("quantity", (F.floor(u("line_key", 33) * 5) + 1).cast("int"))
        .select("order_idx", "line_no", "product_idx", "quantity", "order_date"))

    for name, df in [("customers", customers), ("products", products), ("orders", orders), ("order_items", order_items)]:
        print(f"Building {fq(name)} ...")
        spark.sql(f"DROP TABLE IF EXISTS {fq(name)}")   # also drops the clustering of a previous run
        df.write.saveAsTable(fq(name))
        spark.sql(f"ANALYZE TABLE {fq(name)} COMPUTE STATISTICS FOR ALL COLUMNS")

t0 = time.time()
if REBUILD or not all(spark.catalog.tableExists(fq(t)) for t in TABLES):
    build_tables()
else:
    print("Reusing the tables (set rebuild = yes to start again)")
print(f"Ready in {time.time() - t0:,.0f} s")

**Why `ANALYZE TABLE`.** It computes column statistics for the optimizer, so every query of the lab gets the same join plan and the comparison is fair. On a managed table with Predictive Optimization the platform runs it for you; on tables created a minute ago, we run it once.

## 3. Helpers used by the lab

In [ ]:
def file_stats(t="orders"):
    d = spark.sql(f"DESCRIBE DETAIL {fq(t)}").collect()[0]
    n, size = d["numFiles"], d["sizeInBytes"]
    print(f"{fq(t)}: {n:,} files · {size / 1024**2:,.0f} MB · clustering {d['clusteringColumns']}")

if "BENCH" not in globals(): BENCH = {}   # label -> {median_s, runs}; kept if the setup runs again

def run_fresh(sql_text, tag):
    """Run a query to the end and measure the engine, not the notebook:
    - a unique comment changes the SQL text, so the result cache cannot answer;
    - the noop sink executes the whole plan and throws the rows away (collect() would time the transfer of the result)."""
    spark.sql(f"{sql_text}\n/* lab2402:{tag}:{uuid.uuid4().hex[:8]} */").write.format("noop").mode("overwrite").save()

def benchmark(label, sql_text, runs=5, warmups=1):
    """Honest benchmark: result cache bypassed, warm-up discarded, median of the measured runs."""
    for w in range(warmups):
        run_fresh(sql_text, f"{label}:warmup")
    times = []
    for i in range(runs):
        t0 = time.perf_counter(); run_fresh(sql_text, f"{label}:run"); times.append(time.perf_counter() - t0)
        print(f"  {label} run {i + 1}: {times[-1]:.2f} s")
    times.sort()
    BENCH[label] = dict(median_s=times[len(times) // 2], runs=runs)
    print(f"  {label} median of {runs}: {BENCH[label]['median_s']:.2f} s (warm-up discarded)")

def compare(query_a, query_c):
    """The proof: the medians of the benchmarks, and the same answer from A and C (the rewrite changes nothing)."""
    base = BENCH.get("A", {}).get("median_s")
    for label in sorted(BENCH):
        m = BENCH[label]["median_s"]
        print(f"  {label}: median {m:6.2f} s" + (f"   ({m / base:.0%} of A)" if base else ""))
    rows_a = sorted(tuple(r) for r in spark.sql(query_a).collect())
    rows_c = sorted(tuple(r) for r in spark.sql(query_c).collect())
    print(f"  A and C return the same {len(rows_a)} rows: {rows_a == rows_c}")

## 4. Check before moving on

In [ ]:
for t in TABLES:
    file_stats(t)
display(spark.sql(f"""
    SELECT c.tier AS who,
           COUNT(*) AS orders,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_orders,
           COUNT(DISTINCT o.customer_idx) AS customers
    FROM {fq('orders')} o JOIN {fq('customers')} c ON o.customer_idx = c.customer_idx
    GROUP BY 1 ORDER BY orders DESC"""))

Expected at `medium`: 16 / 16 / 32 / 32 files and no clustering. In the second table, **`guest`** has about 25% of the orders under **one** customer, and **`platinum`** (the whales, 1% of the customers) about 60%.

## Cleanup (optional)
Uncomment when you no longer need the lab. Do not run it as part of *Run all*.

In [ ]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")